# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you in loading and exploring the FAIR² dataset using the `mlcroissant` library. We showcase record set navigation, data extraction, filtering, and normalization steps with programmatic references by `@id`.

### Dataset Source
The dataset is defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata and reference as 'dataset'
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'n/a')}")
print(f"Version: {getattr(metadata, 'version', 'n/a')}")


## 2. Data Overview
Review available record sets, their `@id`s, and fields present in each.

This allows identification of the data structures for correct usage of `@id` references as required by Croissant and `mlcroissant`.

In [ ]:
# List all available record sets by their `@id`
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name}")

# Show fields and columns for each record set
for rs in record_sets:
    print(f"\nRecord Set: {rs.name} (@id: {rs.id})")
    print("Fields:")
    for field in getattr(rs, 'fields', []):
        print(f"  - Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
        columns = getattr(field, 'columns', [])
        if columns:
            print(f"     Columns: {[col.id for col in columns]}")

## 3. Data Extraction
Load tabular data from each record set into DataFrames. The `@id` fields are used for referencing record sets and fields in code for clarity and reproducibility.

In [ ]:
# Prepare DataFrame extraction using only @id fields
dataframes = {}

record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"RecordSets available for extraction: {record_set_ids}")

for rs_id in record_set_ids:
    print(f"Loading records for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"- DataFrame {rs_id} shape: {df.shape}")

# Example: Show column names for the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns in RecordSet '@id'={main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head(3))


## 4. Exploratory Data Analysis (EDA)
We will:
- Select a numeric field (referenced by its `@id`) for filtering and normalization
- Filter out records below a chosen threshold
- Normalize the selected numeric column (z-score)
- Optionally group by a categorical field (`@id`-referenced)


In [ ]:
# Pick numeric and group fields based on the field overview above.
# Example selection (replace with actual @id if needed):
numeric_field_id = None
group_field_id = None

if main_record_set_id:
    # Choose the first numerical field from the fields metadata
    main_rs = next(rs for rs in dataset.metadata.record_sets if rs.id == main_record_set_id)
    for field in getattr(main_rs, 'fields', []):
        if getattr(field, 'data_type', None) in ['Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number']:
            numeric_field_id = field.id
            print(f"Using numeric field: {field.name} (@id: {field.id})")
            break
    # Pick the first non-numeric (likely categorical) for grouping
    for field in getattr(main_rs, 'fields', []):
        if getattr(field, 'data_type', None) not in ['Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number']:
            group_field_id = field.id
            print(f"Will group by: {field.name} (@id: {field.id})")
            break

    if numeric_field_id and numeric_field_id in dataframes[main_record_set_id].columns:
        threshold = 10
        filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
        print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
        display(filtered_df.head(3))

        # Z-score normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))

        # Group by the group field if present
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped mean by '{group_field_id}':")
            display(grouped_df.head(3))
    else:
        print("No appropriate numeric field with data for EDA found.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized values for deeper insight.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='ticks')

if main_record_set_id and numeric_field_id and numeric_field_id in dataframes[main_record_set_id].columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), bins=30, ax=axes[0])
    axes[0].set_title(f"Distribution of {numeric_field_id}")

    # For normalized, check if the EDA code already created the column
    norm_col = f"{numeric_field_id}_normalized"
    if norm_col in dataframes[main_record_set_id].columns:
        sns.histplot(dataframes[main_record_set_id][norm_col].dropna(), bins=30, ax=axes[1])
    else:
        # Normalization on all data
        values = dataframes[main_record_set_id][numeric_field_id].dropna()
        normed = (values - values.mean()) / values.std()
        sns.histplot(normed, bins=30, ax=axes[1])
    axes[1].set_title(f"Normalized {numeric_field_id}")
    plt.show()
else:
    print("Cannot plot: Numeric field missing.")

## 6. Conclusion
In this notebook we loaded the FAIR² dataset by its Croissant schema, explored available record sets and fields using their `@id`, performed extraction to DataFrames, filtered and normalized data, grouped by a categorical attribute, and visualized numeric distributions. All data processing was modular and referenced via Croissant entity `@id`s for reproducibility.

For deeper analyses, repeat or modify EDA steps using other record sets or field `@id`s from the overview section. Explore [mlcroissant documentation](https://mlcommons.github.io/croissant/python/latest/) for more advanced data manipulation!